In [1]:
# The Purpose of this file is to learn to convert PDF files to workable data.
import pandas as pd
import sys
print(sys.executable)

/workspaces/Interest/.venv/bin/python


In [5]:
# Method1: pdfplumber -> extract text from PDF files.
import string

import pdfplumber
# Extract PDF data
text = ""
path = "/workspaces/Interest/.venv/webscraping_play/Fayetteville_PDF.pdf"
with pdfplumber.open(path) as pdf:
   for page in pdf.pages:
      text += page.extract_text()

strings = text.split("\n")

i = 0
n = 0
while i < len(strings):
    strings[i] = strings[i].split()
    for n in range(len(strings[i])):
      if strings[i][n] == "cartons":
         strings[i][n] = 0
      if i > 0:
         strings[i][n] = float(strings[i][n])
      n += 1
    i += 1
df = pd.DataFrame(strings).fillna(0)

df.columns = df.iloc[0]
df = df.drop(0)
df = df.reset_index(drop=True)


df


""


##### Beginning of the pdfconverter

In [2]:
import os

os.path.exists("/workspaces/Interest/.venv/Fayetteville_PDF.pdf")



True

In [3]:
from pdf2image import convert_from_path
import pytesseract
from PIL import Image

pages = convert_from_path("/workspaces/Interest/.venv/Fayetteville_PDF.pdf")

pages[0].save("page1.png", "PNG")

text = pytesseract.image_to_string(Image.open("page1.png"))
print(text)




In [ ]:
from pdf2image import convert_from_path

pdf_path = "/workspaces/Interest/webscraping_play/Fayetteville_PDF.pdf"

pages = convert_from_path(pdf_path, dpi=300)
print(f"Converted {len(pages)} pages to images")

# Save first page for debugging
pages[0].save("page1.png", "PNG")



PDFPageCountError: Unable to get page count.
I/O Error: Couldn't open file '/workspaces/Interest/webscraping_play/Fayetteville_PDF.pdf': No such file or directory.


In [12]:
from pdf2image import convert_from_path
import pytesseract
from PIL import Image

pdf_path = "/workspaces/Interest/.venv/Fayetteville_PDF.pdf"

# Convert all pages
pages = convert_from_path(pdf_path, dpi=300)
print(f"Converted {len(pages)} pages to images")

full_text = ""

for i, page in enumerate(pages):
    # Optional: save each page for debugging
    page.save(f"page_{i+1}.png", "PNG")

    # OCR the page
    text = pytesseract.image_to_string(page)

    print(f"\n--- PAGE {i+1} ---")
    print(text)

    full_text += f"\n\n--- PAGE {i+1} ---\n{text}"


Converted 2 pages to images

--- PAGE 1 ---


--- PAGE 2 ---



In [11]:
from pdf2image import convert_from_path
import pytesseract
from PIL import Image

pdf_path = "/workspaces/Interest/webscraping_play/Fayetteville_PDF.pdf"

# Convert all pages
pages = convert_from_path(pdf_path, dpi=300)
print(f"Converted {len(pages)} pages to images")

full_text = ""

for i, page in enumerate(pages):
    # Optional: save each page for debugging
    page.save(f"page_{i+1}.png", "PNG")

    # OCR the page
    text = pytesseract.image_to_string(page)

    print(f"\n--- PAGE {i+1} ---")
    print(text)

    full_text += f"\n\n--- PAGE {i+1} ---\n{text}"



PDFPageCountError: Unable to get page count.
I/O Error: Couldn't open file '/workspaces/Interest/webscraping_play/Fayetteville_PDF.pdf': No such file or directory.


In [4]:
import pytesseract
from PIL import Image

img = Image.open("page1.png")
text = pytesseract.image_to_string(img)

print(text[:500])  # preview first 500 chars


## Review of Method1:
    > * 'Unnamed' & 'NaN' don't have placeholders.
    > * Because, no placeholders, SHIFTING OF DATA FIELDS.
#### Thoughts:
    > * Great way to work with paragraph entries.
    > * Not great at working with data or tables. 

In [6]:
# Method2: tabula-py -> extract tables from PDF files.
import tabula

dfs = tabula.read_pdf("/workspaces/Interest/webscraping_play/Fayetteville_PDF.pdf", pages="all", multiple_tables=True)
dfs[0].fillna(0)


Error from tabula-java:
Jun 15, 2026 2:03:25 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:25 AM org.apache.pdfbox.filter.FlateFilter decode
SEVERE: FlateFilter: stop reading corrupt stream due to a DataFormatException
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.filter.FlateFilter decode
SEVERE: FlateFilter: stop reading corrupt stream due to a DataFormatException
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:26 AM org.apache.pdfbox.pdfparser.COSParser validateStreamLength
Jun 15, 2026 2:03:26

CalledProcessError: Command '['java', '-Dfile.encoding=UTF8', '-jar', '/usr/local/python/3.12.1/lib/python3.12/site-packages/tabula/tabula-1.0.5-jar-with-dependencies.jar', '--pages', 'all', '--guess', '--format', 'JSON', '/workspaces/Interest/webscraping_play/Fayetteville_PDF.pdf']' returned non-zero exit status 1.

## Method2: tabula 
> *Perfect Data Integrity
> *The file is converted directly into a DataFrame*
> *I am worried this method will not work in more complex files.*

In [3]:
# Method3: Camelot -> extract tables from PDF files.
import camelot

tables = camelot.read_pdf("/workspaces/Interest/webscraping_play/Fayetteville_PDF.pdf")
tables[0].df


xref parsing failed, falling back to object parser: Expected object ID and count, got /b'\xbd' /b'\xef'
Error -3 while decompressing data: incorrect header check: <ContentStream(31): raw=1896, {'Type': /'ObjStm', 'N': 74, 'First': 569, 'Filter': /'FlateDecode', 'Length': 1057}>
Data loss in decompress_corrupted: Error -3 while decompressing data: incorrect header check: bytes 0:1893
Error -3 while decompressing data: incorrect header check: <ContentStream(4): raw=16146, {'Filter': /'FlateDecode', 'Length': 8823}>
Data loss in decompress_corrupted: Error -3 while decompressing data: incorrect header check: bytes 0:4096
/usr/local/python/3.12.1/lib/python3.12/site-packages/camelot/parsers/base.py:272: UserWarning: No tables found on page-1
  if self._document_has_no_text():


IndexError: list index out of range

In [ ]:
# ==========================
# **
# If the PDF is scanned or image
# **
# You'll need OCR (Optical Character Recognition).
# **
# 1) pdf2image -> convert PDF pages to images.
# 2) pytesseract -> extract text from images.
# ***
# ===========================


In [16]:
# Method4: pdf2image + pytesseract -> extract text from scanned PDF files.
from pdf2image import convert_from_path
import pytesseract

pages = convert_from_path("/workspaces/Interest/2026_primary_candidate_list_by_contest_federal_and_state.pdf")

text = ""
for page in pages:
   text += pytesseract.image_to_string(page)

print(text)

PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?